## 01 · Setup

Creates (idempotently) everything the demo needs in `pavan_naidu_catalog.mlflow`:

* the Volume `voice_audio` (input clips, generated TTS audio, TTS model files)
* `voice_golden_utterances`: 12 questions with human-checked transcripts, expected tool calls and facts, and a human-written reference answer (the text every candidate's TTS reads in the oracle condition), plus each clip's Volume path, sha256 and duration, one set of rows per dataset version
* `voice_domain_lexicon`: product names, SKUs and order numbers with their accepted spellings
* the GenAI experiment *Voice Agent Evals*, with traces stored in UC

Before running, make the input clips for the current dataset version (`config.DATASET_VERSION`): v4 (the default) or v2 with `00_input_audio` (Kokoro), v3 with `00_input_audio` for v2 and then `00_noisy_audio`, or v1 from a Mac with `python scripts/make_input_audio.py --upload FEVM`.

**Illustrative data.** Meridian Ortho and every product, SKU and order are made up.

In [ ]:
%pip install -q "mlflow[databricks]>=3.15"
dbutils.library.restartPython()

In [ ]:
import csv, json, os, sys
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))
from voice_evals import config as C
from voice_evals.audio import audio_ref, download_piper_voice
from voice_evals.delta import overwrite, replace_where

DATA = Path(C.__file__).resolve().parent.parent / "data"
print(C.VOLUME_ROOT, f"dataset {C.DATASET_VERSION}: {C.INPUT_DIR}", C.TABLES, sep="\n")

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {C.CATALOG}.{C.SCHEMA}.{C.VOLUME} "
          "COMMENT 'Illustrative data. Voice Agent Evals audio: input clips, TTS output, TTS model files.'")
for d in (C.INPUT_DIR, f"{C.VOLUME_ROOT}/tts", f"{C.VOLUME_ROOT}/models/piper"):
    os.makedirs(d, exist_ok=True)
sorted(os.listdir(C.INPUT_DIR))

In [ ]:
golden = C.load_golden(C.DATASET_VERSION)  # golden_v1.jsonl with that version's rewording (v4)
missing = [g["utterance_id"] for g in golden if not os.path.exists(f"{C.INPUT_DIR}/{g['utterance_id']}.wav")]
assert not missing, f"Make the {C.DATASET_VERSION} input clips first (00_input_audio). Missing: {missing}"

# The manifest names the voice that read each clip (v2 and later); v1 clips use the voice in the golden file.
manifest_path = f"{C.INPUT_DIR}/manifest.json"
manifest = json.load(open(manifest_path)) if os.path.exists(manifest_path) else {}

rows = []
for g in golden:
    ref = audio_ref(f"{C.INPUT_DIR}/{g['utterance_id']}.wav")
    voice = manifest.get(g["utterance_id"], {}).get("voice", g["voice"])
    rows.append({**g, **ref, "voice": voice, "expected_tool_calls": json.dumps(g["expected_tool_calls"]),
                 "synthetic_input": True, "dataset_version": C.DATASET_VERSION})
# One set of rows per dataset version, so earlier runs keep pointing at their own clips.
replace_where(spark, C.TABLES["golden_utterances"], "golden_utterances", rows, f"dataset_version = '{C.DATASET_VERSION}'")
display(spark.table(C.TABLES["golden_utterances"]).groupBy("dataset_version").count())
display(spark.table(C.TABLES["golden_utterances"]).where(f"dataset_version = '{C.DATASET_VERSION}'"))

In [ ]:
with open(DATA / "lexicon_v1.csv", newline="") as f:
    lexicon = [{"term": r["term"], "category": r["category"], "variants": r["variants"].split("|")} for r in csv.DictReader(f)]
overwrite(spark, C.TABLES["domain_lexicon"], "domain_lexicon", lexicon)
display(spark.table(C.TABLES["domain_lexicon"]))

In [ ]:
for voice in C.PIPER_VOICES:
    path = download_piper_voice(C.piper_model_path(voice), C.piper_model_url(voice))
    print(voice, round(os.path.getsize(path) / 1e6, 1), "MB")

In [ ]:
experiment = C.set_experiment()
print(f"Experiment: {experiment.name}")
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Trace location: {getattr(experiment, 'trace_location', None)}")
tables = spark.sql(f"SHOW TABLES IN {C.CATALOG}.{C.SCHEMA}")
display(tables.where(f"tableName LIKE 'voice_%' OR tableName LIKE '%{experiment.experiment_id}%'"))

In [ ]:
dbutils.notebook.exit(json.dumps({"experiment_id": experiment.experiment_id, "utterances": len(rows), "lexicon_terms": len(lexicon)}))